#### Data Loading and Cleaning

In this script, we load the five datasets that will be used for analysis, aggregate based on season, perform feature engineering, and remove missing values

In [ ]:
import pandas as pd 
import numpy as np

#Load necessary data sets
valuations = pd.read_csv("data/player_valuations.csv")
players = pd.read_csv("data/players.csv")
lineups = pd.read_csv("data/game_lineups.csv")
clubs = pd.read_csv("data/clubs.csv")
appearances = pd.read_csv("data/appearances.csv")

C:\Users\megan\AppData\Local\Temp\ipykernel_2928\819163666.py:4: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  lineups = pd.read_csv("data/game_lineups.csv")


In [4]:
#Function to calculate soccer season based on date (typically starts in August)
#Will aggregate player stats by season, to account for data sets having multiple observations per player
def get_season(date):
    month = date.month
    year = date.year
    if month >= 8:  # August to December
        return f"{year}/{year + 1}"
    else:  # January to May
        return f"{year - 1}/{year}"

In [5]:
#Calculate average player market value by season
#Note: records are very sparse before 2004 
valuations['season'] = valuations['date'].apply(lambda x: get_season(pd.to_datetime(x)))
valuations_agg = valuations.groupby(['player_id', 'season'])['market_value_in_eur'].mean().reset_index()
valuations_agg.head()

,player_id,season,market_value_in_eur
0,10,2004/2005,9.333333e+06
1,10,2005/2006,2.166667e+07
2,10,2006/2007,2.300000e+07
3,10,2007/2008,2.000000e+07
4,10,2008/2009,1.800000e+07


In [6]:
#Select relevant fields from players data
players = players[['player_id', 'name', 'date_of_birth', 'position', 'foot']]
players.head()

,player_id,name,date_of_birth,position,foot
0,10,Miroslav Klose,1978-06-09 00:00:00,Attack,right
1,26,Roman Weidenfeller,1980-08-06 00:00:00,Goalkeeper,left
2,65,Dimitar Berbatov,1981-01-30 00:00:00,Attack,NaN
3,77,Lúcio,1978-05-08 00:00:00,Defender,NaN
4,80,Tom Starke,1981-03-18 00:00:00,Goalkeeper,right


In [7]:
#Join valuations and players data sets 
data = pd.merge(valuations_agg, players, on='player_id', how='left')
data.head()

#Calculate player age at each season they have records
data['date_of_birth'] = pd.to_datetime(data['date_of_birth'])
data['season_start_year'] = data['season'].apply(lambda x: int(x.split('/')[0]))
data['age'] = data['season_start_year'] - data['date_of_birth'].dt.year

#Select necessary columns
data = data[['player_id', 'season', 'market_value_in_eur', 'name', 'position', 'foot', 'age']]
data.head()

,player_id,season,market_value_in_eur,name,position,foot,age
0,10,2004/2005,9.333333e+06,Miroslav Klose,Attack,right,26.0
1,10,2005/2006,2.166667e+07,Miroslav Klose,Attack,right,27.0
2,10,2006/2007,2.300000e+07,Miroslav Klose,Attack,right,28.0
3,10,2007/2008,2.000000e+07,Miroslav Klose,Attack,right,29.0
4,10,2008/2009,1.800000e+07,Miroslav Klose,Attack,right,30.0


In [8]:
#Transformations on lineups data
#Calculate number of starter/sub appearances and if they captained per player/season/club
#Note: lineup data is available starting 2013
lineups['season'] = lineups['date'].apply(lambda x: get_season(pd.to_datetime(x)))
lineups['is_starter'] = lineups['type'] != 'substitutes'
lineups['is_sub'] = lineups['type'] == 'substitutes'
lineups['is_captain'] = lineups['team_captain'] == 1
lineups_agg = lineups.groupby(['player_id', 'season', 'club_id']).agg(
    starts=('is_starter', 'sum'),
    subs=('is_sub', 'sum'),
    captains=('is_captain', 'sum')
).reset_index()

lineups_agg.head()

,player_id,season,club_id,starts,subs,captains
0,10,2013/2014,398,24,8,0
1,10,2014/2015,398,26,17,0
2,10,2015/2016,398,19,20,5
3,26,2012/2013,16,1,0,1
4,26,2013/2014,16,42,4,24


In [9]:
#Transformations on appearances data
#Calculate total minutes played, yellow cards, red cards, goals, assists per player/season/club
#Note: appearances data is available starting 2012
appearances['season'] = appearances['date'].apply(lambda x: get_season(pd.to_datetime(x)))
appearances_agg = appearances.groupby(['player_id', 'season', 'player_club_id']).agg(
    total_minutes = ('minutes_played', 'sum'), 
    total_yellow_cards = ('yellow_cards', 'sum'),
    total_red_cards = ('red_cards', 'sum'),
    total_goals = ('goals', 'sum'), 
    total_assists = ('assists', 'sum')
).reset_index()

appearances_agg = appearances_agg.rename(columns={'player_club_id': 'club_id'})
appearances_agg.head()


,player_id,season,club_id,total_minutes,total_yellow_cards,total_red_cards,total_goals,total_assists
0,10,2012/2013,398,2585,8,0,16,3
1,10,2013/2014,398,2220,2,0,8,5
2,10,2014/2015,398,2289,6,0,16,9
3,10,2015/2016,398,1714,3,0,8,8
4,26,2012/2013,16,4491,2,1,0,0


In [10]:
#Select relevant fields from clubs data
clubs = clubs[['club_id', 'name', 'domestic_competition_id']]

#Rename domestic_competiton_id values to be more informative
clubs['league'] = clubs['domestic_competition_id'].replace({
    'TR1': 'Super Lig',
    'IT1': 'Serie A',
    'UKR1': 'Ukrainian Premier League',
    'GB1': 'Premier League',
    'FR1': 'Ligue 1',
    'PO1': 'Primeira Liga',
    'RU1': 'Russian Premier League',
    'ES1': 'La Liga',
    'L1': 'Bundesliga',
    'NL1': 'Eredivisie',
    'GR1': 'Super League Greece', 
    'BE1': 'Belgian Pro League', 
    'DK1': 'Danish Superliga', 
    'SC1': 'Scottish Premiership'
})

#Select necessary columns
clubs = clubs[['club_id', 'name', 'league']]
clubs = clubs.rename(columns={'name': 'club_name'})
clubs.head()


,club_id,club_name,league
0,10,Arminia Bielefeld,Bundesliga
1,10004,Paris Football Club,Ligue 1
2,1003,Leicester City,Premier League
3,1005,Unione Sportiva Lecce,Serie A
4,1010,Watford FC,Premier League


In [11]:
#Join lineups, appearances, and club data to valuations/player data set
#Note: some players may have multiple records for a season if they transferred clubs mid-year
#Inner join means records are only obtained where we have lineup/appearance info for a player in that season
data_2 = pd.merge(data, lineups_agg, on=['player_id', 'season'])
data_3 = pd.merge(data_2, appearances_agg, on = ['player_id', 'season', 'club_id'])
complete_data = pd.merge(data_3, clubs, on = 'club_id')
complete_data.head()


,player_id,season,market_value_in_eur,name,position,foot,age,club_id,starts,subs,captains,total_minutes,total_yellow_cards,total_red_cards,total_goals,total_assists,club_name,league
0,10,2013/2014,1000000.0,Miroslav Klose,Attack,right,35.0,398,24,8,0,2220,2,0,8,5,Società Sportiva Lazio S.p.A.,Serie A
1,10,2014/2015,1000000.0,Miroslav Klose,Attack,right,36.0,398,26,17,0,2289,6,0,16,9,Società Sportiva Lazio S.p.A.,Serie A
2,10,2015/2016,1000000.0,Miroslav Klose,Attack,right,37.0,398,19,20,5,1714,3,0,8,8,Società Sportiva Lazio S.p.A.,Serie A
3,26,2012/2013,4500000.0,Roman Weidenfeller,Goalkeeper,left,32.0,16,1,0,1,4491,2,1,0,0,Borussia Dortmund,Bundesliga
4,26,2013/2014,5000000.0,Roman Weidenfeller,Goalkeeper,left,33.0,16,42,4,24,3765,1,1,0,0,Borussia Dortmund,Bundesliga


#### Further EDA for Complete Data

In [139]:
complete_data.shape

(78249, 18)

In [140]:
#Note: missing values in 'foot' and 'age'; however, makes up less than 2% of data
complete_data.isna().sum()

player_id                 0
season                    0
market_value_in_eur       0
name                      0
position                  0
foot                   1371
age                      40
club_id                   0
starts                    0
subs                      0
captains                  0
total_minutes             0
total_yellow_cards        0
total_red_cards           0
total_goals               0
total_assists             0
club_name                 0
league                    0
dtype: int64

In [141]:
#Frequency of categorical variables
#Note: some values of position are 'Missing' as opposed to NaN; consider removing
print(complete_data['position'].value_counts())
print(complete_data['foot'].value_counts())
print(complete_data['league'].value_counts())

position
Defender      26164
Midfield      23388
Attack        22436
Goalkeeper     6170
Missing          91
Name: count, dtype: int64
foot
right    54585
left     19002
both      3291
Name: count, dtype: int64
league
Serie A                     6962
Belgian Pro League          6531
Russian Premier League      6268
La Liga                     6165
Super Lig                   6034
Ligue 1                     5962
Premier League              5831
Primeira Liga               5707
Super League Greece         5317
Bundesliga                  5272
Eredivisie                  5245
Ukrainian Premier League    5114
Danish Superliga            4026
Scottish Premiership        3815
Name: count, dtype: int64


In [ ]:
#Summary statistics for numeric features
#Note: Age appears symmetric, most other variables are right skewed (some extremely); makes sense in the context of the 
# data because high values are rare for many of these performance metrics 
complete_data[['market_value_in_eur', 'age', 'starts', 'subs', 'captains', 'total_minutes', 'total_yellow_cards', 
                'total_red_cards', 'total_goals', 'total_assists']].describe()

,market_value_in_eur,age,starts,subs,captains,total_minutes,total_yellow_cards,total_red_cards,total_goals,total_assists
count,7.824900e+04,78209.000000,78249.000000,78249.000000,78249.000000,78249.000000,78249.000000,78249.000000,78249.000000,78249.000000
mean,4.023589e+06,25.537815,13.336030,9.370829,1.146686,1223.086289,2.616302,0.066736,1.695638,1.326471
std,9.280158e+06,4.432046,11.825975,8.678618,4.883954,1038.661016,2.903572,0.263950,3.393560,2.294064
min,1.000000e+04,14.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,4.000000e+05,22.000000,2.000000,2.000000,0.000000,270.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000e+06,25.000000,11.000000,7.000000,0.000000,1007.000000,2.000000,0.000000,0.000000,0.000000
75%,3.400000e+06,29.000000,22.000000,14.000000,0.000000,1986.000000,4.000000,0.000000,2.000000,2.000000
max,2.000000e+08,43.000000,56.000000,57.000000,52.000000,5376.000000,23.000000,3.000000,61.000000,31.000000


In [15]:
#Remove records with missing values
final_data = complete_data[(complete_data['foot'].notna()) & (complete_data['age'].notna()) & (complete_data['position'] != 'Missing')]

#Write data to CSV for future reference
final_data.to_csv('final_data.csv', index=False)